# AT1: Perceptron e KNN em Prática

Este notebook implementa, somente com NumPy, três soluções de Aprendizagem de Máquina:

1. Perceptron treinável para triagem de transações.
2. Classificador K-Nearest Neighbors (KNN) para risco de churn.
3. Recomendador de servidores cloud por similaridade espacial.

Todas as etapas são determinísticas e podem ser executadas linearmente do início ao fim.

In [3]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)
print(f"NumPy: {np.__version__}")

NumPy: 2.5.3


## Desafio 1 - Classificação binária com Perceptron

O Perceptron calcula uma combinação linear das características e usa a função degrau. Quando a predição difere do rótulo, os parâmetros são ajustados pela regra de Rosenblatt:

\[
\mathbf{w} \leftarrow \mathbf{w} + \eta (y - \hat{y})\mathbf{x}
\quad\text{e}\quad
b \leftarrow b + \eta (y - \hat{y})
\]

Os pesos e o viés começam em `1.0`, conforme solicitado.

In [4]:
X_treino_fraude = np.array([
    [1.5, 1.0],
    [2.0, 2.0],
    [3.5, 1.5],
    [3.0, 3.0],
    [6.5, 5.0],
    [7.0, 7.0],
    [8.5, 6.0],
    [9.0, 8.0],
])

y_treino_fraude = np.array([0, 0, 0, 0, 1, 1, 1, 1])


def degrau(valor):
    """Retorna 1 para valores nao negativos e 0 caso contrario."""
    return int(valor >= 0)


def treinar_perceptron(X, y, taxa_aprendizado=0.1, epocas=20):
    """Treina um Perceptron usando a regra de ajuste de Rosenblatt."""
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=int)
    pesos = np.ones(X.shape[1], dtype=float)
    vies = 1.0

    for _ in range(epocas):
        for amostra, rotulo in zip(X, y):
            predicao = degrau(np.dot(amostra, pesos) + vies)
            erro = rotulo - predicao
            if erro != 0:
                pesos += taxa_aprendizado * erro * amostra
                vies += taxa_aprendizado * erro

    return pesos, vies


def prever_perceptron(amostra, pesos, vies):
    """Classifica uma nova amostra com os parametros treinados."""
    amostra = np.asarray(amostra, dtype=float)
    return degrau(np.dot(amostra, pesos) + vies)


pesos_fraude, vies_fraude = treinar_perceptron(
    X_treino_fraude,
    y_treino_fraude,
    taxa_aprendizado=0.1,
    epocas=20,
)

transacao_a = np.array([5.0, 4.0])
transacao_b = np.array([7.0, 6.5])
previsao_a = prever_perceptron(transacao_a, pesos_fraude, vies_fraude)
previsao_b = prever_perceptron(transacao_b, pesos_fraude, vies_fraude)

print("Pesos finais:", pesos_fraude)
print("Vies final:", vies_fraude)
print(f"Transacao A {transacao_a}:", "Suspeita (Risco de Fraude)" if previsao_a else "Legitima")
print(f"Transacao B {transacao_b}:", "Suspeita (Risco de Fraude)" if previsao_b else "Legitima")

Pesos finais: [-0.   0.4]
Vies final: -1.2
Transacao A [5. 4.]: Suspeita (Risco de Fraude)
Transacao B [7.  6.5]: Suspeita (Risco de Fraude)


## Desafio 2 - Predição de risco de churn com KNN

A função abaixo calcula todas as distâncias de uma consulta em relação à base de treino de forma vetorizada. O parâmetro `metrica` aceita `"euclidiana"` ou `"manhattan"`, e a classe é escolhida por votação majoritária entre os `k` vizinhos.

In [5]:
X_treino_churn = np.array([
    [2.0, 1.0],
    [3.0, 0.0],
    [5.0, 1.0],
    [6.0, 2.0],
    [15.0, 4.0],
    [18.0, 5.0],
    [20.0, 4.0],
    [22.0, 6.0],
])

y_treino_churn = np.array([0, 0, 0, 0, 1, 1, 1, 1])


def classificar_knn(X, y, ponto, k=3, metrica="euclidiana"):
    """Retorna classe, indices dos vizinhos e distancias ordenadas."""
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=int)
    ponto = np.asarray(ponto, dtype=float)

    if X.ndim != 2 or ponto.shape != (X.shape[1],):
        raise ValueError("O ponto deve ter uma coordenada para cada caracteristica.")
    if not 1 <= k <= len(X):
        raise ValueError("k deve estar entre 1 e o numero de amostras.")
    if metrica == "euclidiana":
        distancias = np.sqrt(np.sum((X - ponto) ** 2, axis=1))
    elif metrica == "manhattan":
        distancias = np.sum(np.abs(X - ponto), axis=1)
    else:
        raise ValueError("Use a metrica 'euclidiana' ou 'manhattan'.")

    indices_vizinhos = np.argsort(distancias)[:k]
    classes_vizinhas = y[indices_vizinhos]
    classes, contagens = np.unique(classes_vizinhas, return_counts=True)
    classe_predita = int(classes[np.argmax(contagens)])
    return classe_predita, indices_vizinhos, distancias[indices_vizinhos]


def exibir_diagnostico_knn(nome, ponto, metrica="euclidiana", k=3):
    classe, indices, distancias = classificar_knn(
        X_treino_churn,
        y_treino_churn,
        ponto,
        k=k,
        metrica=metrica,
    )
    nome_classe = "Alto Risco" if classe == 1 else "Baixo Risco"
    print(f"{nome}: {ponto}")
    print(f"Classe predita: {nome_classe}")
    print(f"Indices dos vizinhos: {indices}")
    print(f"Distancias ({metrica}): {distancias}")
    print()


cliente_1 = np.array([13.0, 10.0])
cliente_2 = np.array([7.0, 1.5])
exibir_diagnostico_knn("Cliente 1", cliente_1, metrica="euclidiana", k=3)
exibir_diagnostico_knn("Cliente 2", cliente_2, metrica="euclidiana", k=3)

print("Exemplo com Manhattan para o Cliente 1:")
exibir_diagnostico_knn("Cliente 1", cliente_1, metrica="manhattan", k=3)

Cliente 1: [13. 10.]
Classe predita: Alto Risco
Indices dos vizinhos: [4 5 6]
Distancias (euclidiana): [6.3246 7.0711 9.2195]

Cliente 2: [7.  1.5]
Classe predita: Baixo Risco
Indices dos vizinhos: [3 2 1]
Distancias (euclidiana): [1.118  2.0616 4.272 ]

Exemplo com Manhattan para o Cliente 1:
Cliente 1: [13. 10.]
Classe predita: Alto Risco
Indices dos vizinhos: [4 5 7]
Distancias (manhattan): [ 8. 10. 13.]



## Desafio 3 - Recomendação de servidores cloud

Como não existem rótulos de classe neste problema, a recomendação é feita ordenando as instâncias pela distância Euclidiana até o perfil solicitado. O enunciado informa `k=2` como quantidade de recomendações.

In [6]:
catalogo_servidores = np.array([
    [2.0, 4.0, 50.0],
    [4.0, 8.0, 100.0],
    [8.0, 16.0, 250.0],
    [16.0, 32.0, 500.0],
    [32.0, 64.0, 1000.0],
    [64.0, 128.0, 2000.0],
])

nomes_servidores = [
    "Micro Instancia Web",
    "Standard App Server",
    "Medium Backend & Cache",
    "Database Enterprise",
    "High Performance Computing",
    "Big Data & AI Training",
]


def recomendar_servidores(catalogo, demanda, k=2):
    """Retorna os indices e as distancias das k instancias mais proximas."""
    catalogo = np.asarray(catalogo, dtype=float)
    demanda = np.asarray(demanda, dtype=float)
    if demanda.shape != (catalogo.shape[1],):
        raise ValueError("A demanda deve ter uma coordenada por atributo.")
    if not 1 <= k <= len(catalogo):
        raise ValueError("k deve estar entre 1 e o numero de servidores.")

    distancias = np.sqrt(np.sum((catalogo - demanda) ** 2, axis=1))
    indices = np.argsort(distancias)[:k]
    return indices, distancias[indices]


demanda_hardware = np.array([12.0, 28.0, 850.0])
indices_recomendados, distancias_recomendadas = recomendar_servidores(
    catalogo_servidores,
    demanda_hardware,
    k=2,
)

print("Perfil de hardware demandado:", demanda_hardware)
for posicao, (indice, distancia) in enumerate(
    zip(indices_recomendados, distancias_recomendadas),
    start=1,
):
    vcpus, memoria, armazenamento = catalogo_servidores[indice]
    print(f"{posicao}o lugar: {nomes_servidores[indice]}")
    print(f"  Especificacoes: {vcpus:g} vCPUs, {memoria:g} GB RAM, {armazenamento:g} GB SSD")
    print(f"  Distancia Euclidiana: {distancia:.4f}")

Perfil de hardware demandado: [ 12.  28. 850.]
1o lugar: High Performance Computing
  Especificacoes: 32 vCPUs, 64 GB RAM, 1000 GB SSD
  Distancia Euclidiana: 155.5506
2o lugar: Database Enterprise
  Especificacoes: 16 vCPUs, 32 GB RAM, 500 GB SSD
  Distancia Euclidiana: 350.0457


## Síntese dos resultados

- O Perceptron aprende uma fronteira linear para separar transações legítimas de suspeitas e classifica as transações A e B.
- O KNN calcula as distâncias sem laços manuais sobre as amostras e apresenta classe, índices e distâncias dos vizinhos.
- O recomendador seleciona as duas VMs geometricamente mais próximas da demanda informada.

A execução deve ser feita em ordem, começando pelo import do NumPy.